In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import datetime
import xgboost

from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize, least_squares, newton, differential_evolution, dual_annealing
from scipy.special import ndtr

from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.metrics import r2_score
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from scipy.optimize import brentq

from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import repeat
import random
import os as _os
import json as _json

In [2]:
### Adapted ML code to output whole dataset (train + test data) instead of only test data

path = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\data_output_final\data_output_2003_top3_qual.dta"
data = pd.read_stata(path)
data = data.drop(["index"], axis = 1)


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# Load Optuna optimised hyperparameters
_bp_path = 'C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\optuna_best_params.json'
_best    = _json.load(open(_bp_path)) if _os.path.exists(_bp_path) else {}

params_xgb_ivol             = _best.get('xgb_ivol',             {})
params_ann_ivol             = _best.get('ann_ivol',              {})
params_lbfgs_ivol           = _best.get('lbfgs_ivol',           {})
params_xgb_price            = _best.get('xgb_price',            {})
params_ann_price            = _best.get('ann_price',             {})
params_lbfgs_price          = _best.get('lbfgs_price',          {})
params_xgb_surface_ivol     = _best.get('xgb_surface_ivol',     {})
params_ann_surface_ivol     = _best.get('ann_surface_ivol',      {})
params_lbfgs_surface_ivol   = _best.get('lbfgs_surface_ivol',   {})
params_xgb_surface_price    = _best.get('xgb_surface_price',    {})
params_ann_surface_price    = _best.get('ann_surface_price',     {})
params_lbfgs_surface_price  = _best.get('lbfgs_surface_price',  {})
params_ann_variable_ivol    = _best.get('ann_variable_ivol',     {})
params_lbfgs_variable_ivol  = _best.get('lbfgs_variable_ivol',  {})
params_ann_variable_price   = _best.get('ann_variable_price',    {})
params_lbfgs_variable_price = _best.get('lbfgs_variable_price', {})

if _best:
    print("Loaded Optuna params for:", list(_best.keys()))
else:
    print("optuna_best_params.json not found — using hardcoded defaults")

Loaded Optuna params for: ['xgb_ivol', 'ann_ivol', 'lbfgs_ivol', 'xgb_price', 'ann_price', 'lbfgs_price', 'xgb_surface_ivol', 'ann_surface_ivol', 'lbfgs_surface_ivol', 'xgb_surface_price', 'ann_surface_price', 'lbfgs_surface_price', 'ann_variable_ivol', 'lbfgs_variable_ivol', 'ann_variable_price', 'lbfgs_variable_price']


Implied Volatility Models

In [6]:
##### AGBoost ML calibration and estimation functions:

def XGB_function(data, x_cols, y_cols, params=None, compute_importance=False):
    
    X, Y = data[x_cols].copy(), data[y_cols].copy()

    dates  = data["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * 0.7)]
    train_mask = data["date"] < cutoff
    X_train, X_test = X[train_mask].copy(), X[~train_mask].copy()
    Y_train, Y_test = Y[train_mask].copy(), Y[~train_mask].copy()

    train_dates = data.loc[train_mask, "date"].sort_values().unique()
    val_cutoff  = train_dates[int(len(train_dates) * 0.9)]
    val_mask    = train_mask & (data["date"] >= val_cutoff)
    X_val,   Y_val   = X[val_mask].copy(),              Y[val_mask].copy()
    X_train, Y_train = X[train_mask & ~val_mask].copy(), Y[train_mask & ~val_mask].copy()

    X_train_encoded_ivol = X_train
    X_val_encoded_ivol = X_val
    X_test_encoded_ivol = X_test
    Y_train_ivol = Y_train
    Y_val_ivol = Y_val
    Y_test_ivol = Y_test


    _p = params or {}
    model_xgb_ivol = xgboost.XGBRegressor(
        gamma         = _p.get('gamma',         1),
        learning_rate = _p.get('learning_rate', 0.1),
        max_depth     = _p.get('max_depth',     20),
        n_estimators  = _p.get('n_estimators',  200),
        subsample     = _p.get('subsample',     1),
        reg_lambda    = _p.get('reg_lambda',    1),
        eval_metric='rmse', tree_method='hist', device='cuda',
    )

    model_xgb_ivol.fit(X_train_encoded_ivol, Y_train_ivol, verbose=False,
                       eval_set=[(X_val_encoded_ivol, Y_val_ivol)])

    y_pred_xgb_ivol       = model_xgb_ivol.predict(X_test_encoded_ivol)
    y_pred_xgb_ivol_val   = model_xgb_ivol.predict(X_val_encoded_ivol)
    y_pred_xgb_ivol_train = model_xgb_ivol.predict(X_train_encoded_ivol)

    rmse_xgb_ivol = np.sqrt(mean_squared_error(Y_test_ivol, y_pred_xgb_ivol))
    mse_xgb_ivol = mean_squared_error(Y_test_ivol, y_pred_xgb_ivol)
    mape_xgb_ivol = mean_absolute_percentage_error(Y_test_ivol, y_pred_xgb_ivol)
    rsquared_xgb_ivol = r2_score(Y_test_ivol, y_pred_xgb_ivol)


    print("mse:", mse_xgb_ivol)
    print("rmse:", rmse_xgb_ivol)
    print("mape:", mape_xgb_ivol)
    print("rsquared:", rsquared_xgb_ivol)

    importance_df = None
    if compute_importance:
        xgb_gain = model_xgb_ivol.feature_importances_
        perm_xgb = permutation_importance(
            model_xgb_ivol, X_test_encoded_ivol.values.astype(np.float32),
            Y_test_ivol.values.ravel().astype(np.float32),
            n_repeats=10, scoring='neg_mean_squared_error', random_state=42
        )
        importance_df = pd.DataFrame({
            'feature': x_cols,
            'xgb_gain_importance': xgb_gain,
            'xgb_perm_mean': perm_xgb.importances_mean,
            'xgb_perm_std': perm_xgb.importances_std,
        }).set_index('feature').sort_values('xgb_perm_mean', ascending=False)
        print("  Feature importance computed")


    output_xgb_ivol = pd.concat([X_test_encoded_ivol, 
                                 Y_test_ivol.set_axis(X_test_encoded_ivol.index), 
                                 pd.DataFrame(y_pred_xgb_ivol, columns = ["output_xgb"]).set_axis(X_test_encoded_ivol.index)], axis=1)
    output_xgb_ivol_val = pd.concat([X_val_encoded_ivol,
                                     Y_val_ivol.set_axis(X_val_encoded_ivol.index),
                                     pd.DataFrame(y_pred_xgb_ivol_val, columns=["output_xgb"]).set_axis(X_val_encoded_ivol.index)], axis=1)
    output_xgb_ivol_train = pd.concat([X_train_encoded_ivol, 
                                      Y_train_ivol.set_axis(X_train_encoded_ivol.index), 
                                      pd.DataFrame(y_pred_xgb_ivol_train, columns = ["output_xgb"]).set_axis(X_train_encoded_ivol.index)], axis=1)

    output_xgb_ivol["out_of_sample"] = 1
    output_xgb_ivol_val["out_of_sample"] = 0
    output_xgb_ivol_train["out_of_sample"] = 0

    output_xgb_ivol_full = pd.concat([output_xgb_ivol_train, output_xgb_ivol_val, output_xgb_ivol], axis = 0)

    data_clean = data.drop(columns = [col for col in data.columns if col in output_xgb_ivol_full.columns])
    data_out = pd.concat([data_clean, output_xgb_ivol_full], axis = 1)

    data_out["mse_optim"] = mse_xgb_ivol
    data_out["rmse_optim"] = rmse_xgb_ivol
    data_out["mape_optim"] = mape_xgb_ivol
    data_out["rsquared_optim"] = rsquared_xgb_ivol

    if compute_importance:
        return data_out, importance_df
    return data_out

In [9]:
# ML data prep, modelling option implied volatility
set_seed(42)
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['impl_volatility']


output_xgb_ivol, importance_xgb_ivol = XGB_function(data, selected_columns_x, selected_columns_y, params=params_xgb_ivol, compute_importance=True)
output_xgb_ivol = output_xgb_ivol.rename(columns={"output_xgb":"impl_vol_xgb"})
output_xgb_ivol

c:\Users\chril\torch_gpu\Lib\site-packages\xgboost\core.py:751: UserWarning: [23:02:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


mse: 0.005232819821685553
rmse: 0.0723382320884714
mape: 0.17130212485790253
rsquared: 0.7205931544303894
  Feature importance computed


,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,ddelta,dgamma,...,stock_price,realized_vol,moneyness,impl_volatility,impl_vol_xgb,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2002-01-02,2002-01-19,C,105.0,0.2,0.30,34.0,59.0,0.134418,0.057799,...,100.73,0.004923,0.041517,0.183356,0.151205,0,0.005233,0.072338,0.171302,0.720593
1,2002-01-02,2002-01-19,C,106.0,0.1,0.15,6.0,260.0,0.078105,0.040581,...,100.73,0.004923,0.050995,0.176133,0.151205,0,0.005233,0.072338,0.171302,0.720593
2,2002-01-02,2002-01-19,C,90.0,10.5,11.00,0.0,160.0,0.974934,0.009727,...,100.73,0.004923,-0.112634,0.284675,0.292717,0,0.005233,0.072338,0.171302,0.720593
3,2002-01-02,2002-01-19,C,91.0,9.6,10.00,0.0,30.0,0.957577,0.014793,...,100.73,0.004923,-0.101584,0.293254,0.292717,0,0.005233,0.072338,0.171302,0.720593
4,2002-01-02,2002-01-19,C,92.0,8.6,9.00,0.0,612.0,0.954278,0.017296,...,100.73,0.004923,-0.090655,0.266864,0.265531,0,0.005233,0.072338,0.171302,0.720593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166267,2002-12-31,2003-09-20,P,925.0,101.3,101.50,0.0,996.0,-0.553557,0.002103,...,879.82,0.234007,0.050076,0.248388,0.309065,1,0.005233,0.072338,0.171302,0.720593
166268,2002-12-31,2003-09-20,P,950.0,116.0,116.20,0.0,639.0,-0.607127,0.002091,...,879.82,0.234007,0.076745,0.242036,0.309065,1,0.005233,0.072338,0.171302,0.720593
166269,2002-12-31,2003-09-20,P,975.0,132.2,132.40,0.0,1256.0,-0.659065,0.002032,...,879.82,0.234007,0.102720,0.236583,0.309065,1,0.005233,0.072338,0.171302,0.720593
166270,2002-12-31,2003-09-20,P,900.0,88.1,88.30,1.0,1941.0,-0.499880,0.002068,...,879.82,0.234007,0.022677,0.255549,0.309065,1,0.005233,0.072338,0.171302,0.720593


In [6]:
##### ANN torch calibration and estimation function with code data pipeline correction:
def ANN_function_torch_LBFGS(data, x_cols, y_cols, ignore, batch = 230000, log_y = False, params=None, compute_importance=False):

    X, Y = data[x_cols].copy(), data[y_cols].copy()
    
    dates  = data["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * 0.7)]
    train_mask = data["date"] < cutoff
    X_train, X_test = X[train_mask].copy(), X[~train_mask].copy()
    Y_train, Y_test = Y[train_mask].copy(), Y[~train_mask].copy()

    train_dates = data.loc[train_mask, "date"].sort_values().unique()
    val_cutoff  = train_dates[int(len(train_dates) * 0.9)]
    val_mask    = train_mask & (data["date"] >= val_cutoff)
    X_val,   Y_val   = X[val_mask].copy(),              Y[val_mask].copy()
    X_train, Y_train = X[train_mask & ~val_mask].copy(), Y[train_mask & ~val_mask].copy()


    scaler_X_ivol = StandardScaler()

    non_categ_cols_ivol = [col for col in X_train.columns if col not in ignore]
    X_train_encoded_ivol_std = X_train
    X_test_encoded_ivol_std = X_test
    X_val_encoded_ivol_std = X_val
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.fit_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])
    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.transform(X_test_encoded_ivol_std[non_categ_cols_ivol])
    X_val_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.transform(X_val_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_train_ivol_std = np.log(Y_train)
        Y_test_ivol_std  = np.log(Y_test)
        Y_val_ivol_std   = np.log(Y_val)
    else:
        Y_train_ivol_std = Y_train
        Y_test_ivol_std  = Y_test
        Y_val_ivol_std   = Y_val


    _p           = params or {}
    n_layers     = _p.get('n_layers',     3)
    n_units      = _p.get('n_units',      64)
    max_iter     = _p.get('max_iter',     100)
    history_size = _p.get('history_size', 10)


    class MLP(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            layers = [nn.Linear(n_features, n_units), nn.GELU()]
            for _ in range(n_layers - 1):
                layers += [nn.Linear(n_units, n_units), nn.GELU()]
            layers.append(nn.Linear(n_units, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x) if log_y else torch.nn.functional.softplus(self.net(x))

    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


    model = MLP(len(X_train_encoded_ivol_std.columns)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.LBFGS(
        model.parameters(),
        max_iter=max_iter,
        history_size=history_size,
        line_search_fn="strong_wolfe",
        tolerance_grad=1e-7,
        tolerance_change=1e-9,
    )


    def closure():
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            return loss




    n = len(X_train_encoded_ivol_std)

    X_val_t = torch.as_tensor(X_val_encoded_ivol_std.values, device=device, dtype=torch.float32)
    Y_val_t = torch.as_tensor(Y_val_ivol_std.values, device=device, dtype=torch.float32)

    best_val_loss = float("inf")
    patience = 15
    counter = 0
    for epoch in range(50):
        for i in range(0, n, batch):
            model.train()
            end = min(i + batch, n)
            total_loss = 0
            xb = torch.as_tensor(X_train_encoded_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)
            yb = torch.as_tensor(Y_train_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)
            
            loss_temp = optimizer.step(closure)
            total_loss += loss_temp.detach().item()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), Y_val_t)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break
        print(f"Run {epoch:03d} | Training Loss: {total_loss:.4f}")
        print(f"Run {epoch:03d} | Best Validation Loss: {best_val_loss:.4f}")

    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        
        X_test_encoded_ivol_std_t = torch.from_numpy(X_test_encoded_ivol_std.values.astype(np.float32)).to(device)
        X_val_encoded_ivol_std_t = torch.from_numpy(X_val_encoded_ivol_std.values.astype(np.float32)).to(device)
        X_train_encoded_ivol_std_t = torch.from_numpy(X_train_encoded_ivol_std.values.astype(np.float32)).to(device)
        y_pred_ann_ivol = model(X_test_encoded_ivol_std_t).cpu().numpy()
        y_pred_ann_ivol_val = model(X_val_encoded_ivol_std_t).cpu().numpy()
        y_pred_ann_ivol_train = model(X_train_encoded_ivol_std_t).cpu().numpy()

    if log_y:
        y_pred_ann_ivol       = np.exp(y_pred_ann_ivol)
        y_pred_ann_ivol_val   = np.exp(y_pred_ann_ivol_val)
        y_pred_ann_ivol_train = np.exp(y_pred_ann_ivol_train)

    rmse_ann_ivol = np.sqrt(mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol))
    mse_ann_ivol = mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol)
    mape_ann_ivol = mean_absolute_percentage_error(Y_test.values.ravel(), y_pred_ann_ivol)
    rsquared_ann_ivol = r2_score(Y_test.values.ravel(), y_pred_ann_ivol)


    print("mse:", mse_ann_ivol)
    print("rmse:", rmse_ann_ivol)
    print("mape:", mape_ann_ivol)
    print("rsquared:", rsquared_ann_ivol)

    importance_df = None
    if compute_importance:
        class _TorchPredictor:
            def __init__(self, _model, _device):
                self._model = _model
                self._device = _device
            def predict(self, X):
                with torch.no_grad():
                    X_t = torch.tensor(X.astype(np.float32), device=self._device)
                    return self._model(X_t).cpu().numpy().ravel()

        predictor = _TorchPredictor(model, device)
        perm_ann = permutation_importance(
            predictor, X_test_encoded_ivol_std.values.astype(np.float32),
            Y_test_ivol_std.values.ravel().astype(np.float32),
            n_repeats=10, scoring='neg_mean_squared_error', random_state=42
        )
        importance_df = pd.DataFrame({
            'feature': x_cols,
            'ann_lbfgs_perm_mean': perm_ann.importances_mean,
            'ann_lbfgs_perm_std': perm_ann.importances_std,
        }).set_index('feature').sort_values('ann_lbfgs_perm_mean', ascending=False)
        print("  Feature importance computed")

    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_test_encoded_ivol_std[non_categ_cols_ivol])
    X_val_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_val_encoded_ivol_std[non_categ_cols_ivol])
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_test_ivol_std  = np.exp(Y_test_ivol_std)
        Y_val_ivol_std   = np.exp(Y_val_ivol_std)
        Y_train_ivol_std = np.exp(Y_train_ivol_std)

    output_ann_ivol = pd.concat([X_test_encoded_ivol_std, 
                                 Y_test_ivol_std.set_axis(X_test_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol, columns = ["output_ann"]).set_axis(X_test_encoded_ivol_std.index)], axis=1)
    output_ann_ivol_val = pd.concat([X_val_encoded_ivol_std, 
                                 Y_val_ivol_std.set_axis(X_val_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol_val, columns = ["output_ann"]).set_axis(X_val_encoded_ivol_std.index)], axis=1)
    output_ann_ivol_train = pd.concat([X_train_encoded_ivol_std, 
                                 Y_train_ivol_std.set_axis(X_train_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol_train, columns = ["output_ann"]).set_axis(X_train_encoded_ivol_std.index)], axis=1)


    output_ann_ivol["out_of_sample"] = 1
    output_ann_ivol_val["out_of_sample"] = 0
    output_ann_ivol_train["out_of_sample"] = 0

    output_ann_ivol_full = pd.concat([output_ann_ivol_train, output_ann_ivol_val, output_ann_ivol], axis = 0)

    data_clean = data.drop(columns = [col for col in data.columns if col in output_ann_ivol_full.columns])
    data_out = pd.concat([data_clean, output_ann_ivol_full], axis = 1)

    data_out["mse_optim"] = mse_ann_ivol
    data_out["rmse_optim"] = rmse_ann_ivol
    data_out["mape_optim"] = mape_ann_ivol
    data_out["rsquared_optim"] = rsquared_ann_ivol

    if compute_importance:
        return data_out, importance_df
    return data_out

In [10]:
# ML data prep, modelling option implied volatility
set_seed(42)
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['impl_volatility']

ignore = ["cp_flag_one"]
output_ann_ivol_lbfgs = ANN_function_torch_LBFGS(data, selected_columns_x, selected_columns_y, ignore, params=params_lbfgs_ivol, compute_importance=False)
output_ann_ivol_lbfgs = output_ann_ivol_lbfgs.rename(columns={"output_ann":"impl_vol_ann"})
output_ann_ivol_lbfgs

Run 000 | Training Loss: 0.1918
Run 000 | Best Validation Loss: 0.0036
Run 001 | Training Loss: 0.0017
Run 001 | Best Validation Loss: 0.0030
Run 002 | Training Loss: 0.0013
Run 002 | Best Validation Loss: 0.0023
Run 003 | Training Loss: 0.0010
Run 003 | Best Validation Loss: 0.0022
Run 004 | Training Loss: 0.0009
Run 004 | Best Validation Loss: 0.0022
Run 005 | Training Loss: 0.0008
Run 005 | Best Validation Loss: 0.0022
Run 006 | Training Loss: 0.0008
Run 006 | Best Validation Loss: 0.0022
Run 007 | Training Loss: 0.0008
Run 007 | Best Validation Loss: 0.0021
Run 008 | Training Loss: 0.0008
Run 008 | Best Validation Loss: 0.0021
Run 009 | Training Loss: 0.0007
Run 009 | Best Validation Loss: 0.0021
Run 010 | Training Loss: 0.0007
Run 010 | Best Validation Loss: 0.0021
Run 011 | Training Loss: 0.0007
Run 011 | Best Validation Loss: 0.0021
Run 012 | Training Loss: 0.0006
Run 012 | Best Validation Loss: 0.0021
Run 013 | Training Loss: 0.0006
Run 013 | Best Validation Loss: 0.0021
Run 01

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,ddelta,dgamma,...,stock_price,realized_vol,moneyness,impl_volatility,impl_vol_ann,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2002-01-02,2002-01-19,C,105.0,0.2,0.30,34.0,59.0,0.134418,0.057799,...,100.73,0.004923,0.041517,0.183356,0.166383,0,0.020693,0.143851,0.292167,-0.104918
1,2002-01-02,2002-01-19,C,106.0,0.1,0.15,6.0,260.0,0.078105,0.040581,...,100.73,0.004923,0.050995,0.176133,0.170794,0,0.020693,0.143851,0.292167,-0.104918
2,2002-01-02,2002-01-19,C,90.0,10.5,11.00,0.0,160.0,0.974934,0.009727,...,100.73,0.004923,-0.112634,0.284675,0.332337,0,0.020693,0.143851,0.292167,-0.104918
3,2002-01-02,2002-01-19,C,91.0,9.6,10.00,0.0,30.0,0.957577,0.014793,...,100.73,0.004923,-0.101584,0.293254,0.309126,0,0.020693,0.143851,0.292167,-0.104918
4,2002-01-02,2002-01-19,C,92.0,8.6,9.00,0.0,612.0,0.954278,0.017296,...,100.73,0.004923,-0.090655,0.266864,0.287186,0,0.020693,0.143851,0.292167,-0.104918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166267,2002-12-31,2003-09-20,P,925.0,101.3,101.50,0.0,996.0,-0.553557,0.002103,...,879.82,0.234007,0.050076,0.248388,0.382163,1,0.020693,0.143851,0.292167,-0.104918
166268,2002-12-31,2003-09-20,P,950.0,116.0,116.20,0.0,639.0,-0.607127,0.002091,...,879.82,0.234007,0.076745,0.242036,0.378146,1,0.020693,0.143851,0.292167,-0.104918
166269,2002-12-31,2003-09-20,P,975.0,132.2,132.40,0.0,1256.0,-0.659065,0.002032,...,879.82,0.234007,0.102720,0.236583,0.374529,1,0.020693,0.143851,0.292167,-0.104918
166270,2002-12-31,2003-09-20,P,900.0,88.1,88.30,1.0,1941.0,-0.499880,0.002068,...,879.82,0.234007,0.022677,0.255549,0.386536,1,0.020693,0.143851,0.292167,-0.104918


Price Models

In [11]:
# XGBoost modelling option price
set_seed(42)
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['price']

output_xgb_price, importance_xgb_price = XGB_function(data, selected_columns_x, selected_columns_y, params=params_xgb_price, compute_importance=True)
output_xgb_price = output_xgb_price.rename(columns={"output_xgb":"price_xgb"})
output_xgb_price

mse: 195.66627502441406
rmse: 13.988076173098788
mape: 0.3618861734867096
rsquared: 0.9787154197692871
  Feature importance computed


,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,stock_price,realized_vol,moneyness,price,price_xgb,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2002-01-02,2002-01-19,C,105.0,0.2,0.30,34.0,59.0,0.183356,0.134418,...,100.73,0.004923,0.041517,0.250,0.506912,0,195.666275,13.988076,0.361886,0.978715
1,2002-01-02,2002-01-19,C,106.0,0.1,0.15,6.0,260.0,0.176133,0.078105,...,100.73,0.004923,0.050995,0.125,0.181007,0,195.666275,13.988076,0.361886,0.978715
2,2002-01-02,2002-01-19,C,90.0,10.5,11.00,0.0,160.0,0.284675,0.974934,...,100.73,0.004923,-0.112634,10.750,11.365326,0,195.666275,13.988076,0.361886,0.978715
3,2002-01-02,2002-01-19,C,91.0,9.6,10.00,0.0,30.0,0.293254,0.957577,...,100.73,0.004923,-0.101584,9.800,10.234215,0,195.666275,13.988076,0.361886,0.978715
4,2002-01-02,2002-01-19,C,92.0,8.6,9.00,0.0,612.0,0.266864,0.954278,...,100.73,0.004923,-0.090655,8.800,9.219738,0,195.666275,13.988076,0.361886,0.978715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166267,2002-12-31,2003-09-20,P,925.0,101.3,101.50,0.0,996.0,0.248388,-0.553557,...,879.82,0.234007,0.050076,101.400,133.029755,1,195.666275,13.988076,0.361886,0.978715
166268,2002-12-31,2003-09-20,P,950.0,116.0,116.20,0.0,639.0,0.242036,-0.607127,...,879.82,0.234007,0.076745,116.100,146.144089,1,195.666275,13.988076,0.361886,0.978715
166269,2002-12-31,2003-09-20,P,975.0,132.2,132.40,0.0,1256.0,0.236583,-0.659065,...,879.82,0.234007,0.102720,132.300,161.585312,1,195.666275,13.988076,0.361886,0.978715
166270,2002-12-31,2003-09-20,P,900.0,88.1,88.30,1.0,1941.0,0.255549,-0.499880,...,879.82,0.234007,0.022677,88.200,120.263580,1,195.666275,13.988076,0.361886,0.978715


In [7]:
# Torch ANN with lbfgs modelling option price
set_seed(42)
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['price']

ignore = ["cp_flag_one"]
output_ann_price_lbfgs = ANN_function_torch_LBFGS(data, selected_columns_x, selected_columns_y, ignore, log_y=True, params=params_lbfgs_price, compute_importance=False)
output_ann_price_lbfgs = output_ann_price_lbfgs.rename(columns={"output_ann":"price_ann"})

C:\Users\chril\AppData\Local\Temp\ipykernel_14936\4038091348.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  Y_val_t = torch.as_tensor(Y_val_ivol_std.values, device=device, dtype=torch.float32)


Run 000 | Training Loss: 12.6979
Run 000 | Best Validation Loss: 0.1281
Run 001 | Training Loss: 0.0798
Run 001 | Best Validation Loss: 0.1005
Run 002 | Training Loss: 0.0541
Run 002 | Best Validation Loss: 0.0968
Run 003 | Training Loss: 0.0491
Run 003 | Best Validation Loss: 0.0949
Run 004 | Training Loss: 0.0470
Run 004 | Best Validation Loss: 0.0910
Run 005 | Training Loss: 0.0448
Run 005 | Best Validation Loss: 0.0874
Run 006 | Training Loss: 0.0427
Run 006 | Best Validation Loss: 0.0774
Run 007 | Training Loss: 0.0395
Run 007 | Best Validation Loss: 0.0770
Run 008 | Training Loss: 0.0354
Run 008 | Best Validation Loss: 0.0575
Run 009 | Training Loss: 0.0309
Run 009 | Best Validation Loss: 0.0532
Run 010 | Training Loss: 0.0273
Run 010 | Best Validation Loss: 0.0530
Run 011 | Training Loss: 0.0251
Run 011 | Best Validation Loss: 0.0495
Run 012 | Training Loss: 0.0235
Run 012 | Best Validation Loss: 0.0495
Run 013 | Training Loss: 0.0219
Run 013 | Best Validation Loss: 0.0495
Run 0

Volatility Surface Estimation

In [6]:
##### AGBoost ML calibration and estimation functions:

def XGB_function_surface(data, x_cols, y_cols, params=None):
    
    X, Y = data[x_cols].copy(), data[y_cols].copy()

    label_encoders_ivol = {}
    categorical_columns = []
    for col in categorical_columns:
        label_encoder = LabelEncoder()
        X[col] = label_encoder.fit_transform(X[col])
        label_encoders_ivol[col] = label_encoder

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size= 0.7, shuffle = True, random_state = 0)
    
    X_train_encoded_ivol = X_train
    X_test_encoded_ivol = X_test
    Y_train_ivol = Y_train
    Y_test_ivol = Y_test


    _p = params or {}
    model_xgb_ivol = xgboost.XGBRegressor(
        gamma         = _p.get('gamma',         1),
        learning_rate = _p.get('learning_rate', 0.1),
        max_depth     = _p.get('max_depth',     20),
        n_estimators  = _p.get('n_estimators',  200),
        subsample     = _p.get('subsample',     1),
        reg_lambda    = _p.get('reg_lambda',    1),
        eval_metric='rmse', tree_method='hist', device='cuda',
    )

    model_xgb_ivol.fit(X_train_encoded_ivol, Y_train_ivol, verbose = False, eval_set = [(X_test_encoded_ivol, Y_test_ivol)])

    y_pred_xgb_ivol = model_xgb_ivol.predict(X_test_encoded_ivol)
    y_pred_xgb_ivol_train = model_xgb_ivol.predict(X_train_encoded_ivol)

    rmse_xgb_ivol = np.sqrt(mean_squared_error(Y_test_ivol, y_pred_xgb_ivol))
    mse_xgb_ivol = mean_squared_error(Y_test_ivol, y_pred_xgb_ivol)
    mape_xgb_ivol = mean_absolute_percentage_error(Y_test_ivol, y_pred_xgb_ivol)
    rsquared_xgb_ivol = r2_score(Y_test_ivol, y_pred_xgb_ivol)


    print("mse:", mse_xgb_ivol)
    print("rmse:", rmse_xgb_ivol)
    print("mape:", mape_xgb_ivol)
    print("rsquared:", rsquared_xgb_ivol)


    for col in categorical_columns:
        X_test_encoded_ivol[col] = label_encoders_ivol[col].inverse_transform(X_test_encoded_ivol[col])
        X_train_encoded_ivol[col] = label_encoders_ivol[col].inverse_transform(X_train_encoded_ivol[col])


    output_xgb_ivol = pd.concat([X_test_encoded_ivol, 
                                 Y_test_ivol.set_axis(X_test_encoded_ivol.index), 
                                 pd.DataFrame(y_pred_xgb_ivol, columns = ["output_xgb"]).set_axis(X_test_encoded_ivol.index)], axis=1)
    output_xgb_ivol_train = pd.concat([X_train_encoded_ivol, 
                                      Y_train_ivol.set_axis(X_train_encoded_ivol.index), 
                                      pd.DataFrame(y_pred_xgb_ivol_train, columns = ["output_xgb"]).set_axis(X_train_encoded_ivol.index)], axis=1)

    output_xgb_ivol["out_of_sample"] = 1
    output_xgb_ivol_train["out_of_sample"] = 0

    output_xgb_ivol_full = pd.concat([output_xgb_ivol_train, output_xgb_ivol], axis = 0)

    data_clean = data.drop(columns = [col for col in data.columns if col in output_xgb_ivol_full.columns])
    data_out = pd.concat([data_clean, output_xgb_ivol_full], axis = 1)

    data_out["mse_optim"] = mse_xgb_ivol
    data_out["rmse_optim"] = rmse_xgb_ivol
    data_out["mape_optim"] = mape_xgb_ivol
    data_out["rsquared_optim"] = rsquared_xgb_ivol

    return(data_out)

In [7]:
# Group data on day and cusip (underlying)
set_seed(42)
grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['impl_volatility']

streams = {}
results = {}
for i, group in grouped_data:
    streams[i] = torch.cuda.Stream()
    results[i] = torch.empty(1,device="cuda")

for i, group in grouped_data:
    stream = streams[i]
    with torch.cuda.stream(stream):
        print(group["date"].unique())
        
        temp = XGB_function_surface(group, selected_columns_x, selected_columns_y, params=params_xgb_surface_ivol)
        results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


for stream in streams.values():
    stream.synchronize()

surface_xgb_ivol_data = pd.concat(results.values())
surface_xgb_ivol_data

_oos = surface_xgb_ivol_data[surface_xgb_ivol_data["out_of_sample"] == 1]
print("explained variance score:", explained_variance_score(np.array(np.exp(_oos["impl_volatility"])-1), _oos["output_xgb"]))
print("mse:", mean_squared_error(_oos["impl_volatility"], _oos["output_xgb"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["impl_volatility"], _oos["output_xgb"])))
print("mape:", mean_absolute_percentage_error(_oos["impl_volatility"], _oos["output_xgb"]))
print("rsquared:", r2_score(_oos["impl_volatility"], _oos["output_xgb"]))

surface_xgb_ivol_data = surface_xgb_ivol_data.rename(columns={"output_xgb":"impl_vol_xgb"})
surface_xgb_ivol_data


<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]


c:\Users\chril\torch_gpu\Lib\site-packages\xgboost\core.py:751: UserWarning: [16:37:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


mse: 0.0004646865709219128
rmse: 0.021556589965064345
mape: 0.03863424062728882
rsquared: 0.9549427628517151
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 0.0002547635813243687
rmse: 0.015961315150211423
mape: 0.03459857776761055
rsquared: 0.9756075739860535
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 0.000237670901697129
rmse: 0.015416578793530327
mape: 0.03210277482867241
rsquared: 0.9711090922355652
<DatetimeArray>
['2003-01-07 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 0.00041184647125191987
rmse: 0.020294000868530577
mape: 0.03296791389584541
rsquared: 0.9670687317848206
<DatetimeArray>
['2003-01-08 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 0.00020458016660995781
rmse: 0.014303152331215584
mape: 0.029570432379841805
rsquared: 0.979239284992218
<DatetimeArray>
['2003-01-09 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 0.000355210795532912
rmse: 0.018847036783879636
mape: 0.03192781284451485
rsquared: 

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,ddelta,dgamma,...,stock_price,realized_vol,moneyness,impl_volatility,impl_vol_xgb,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.309251,0.047734,...,86.08,0.030519,0.044533,0.233642,0.230018,0,0.000465,0.021557,0.038634,0.954943
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.256302,0.045347,...,86.08,0.030519,0.055582,0.224751,0.230018,1,0.000465,0.021557,0.038634,0.954943
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.216934,0.041150,...,86.08,0.030519,0.066511,0.225981,0.230018,1,0.000465,0.021557,0.038634,0.954943
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.139982,0.032262,...,86.08,0.030519,0.088018,0.218488,0.223365,0,0.000465,0.021557,0.038634,0.954943
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.361403,0.037649,...,86.08,0.048298,0.044533,0.250788,0.241419,0,0.000465,0.021557,0.038634,0.954943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,-0.191240,0.001474,...,1111.92,0.093558,-0.131406,0.196156,0.201530,1,0.000180,0.013406,0.037532,0.942798
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,-0.220435,0.001657,...,1111.92,0.093558,-0.111101,0.189753,0.192857,0,0.000180,0.013406,0.037532,0.942798
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,-0.111764,0.000908,...,1111.92,0.093558,-0.211449,0.222397,0.234245,1,0.000180,0.013406,0.037532,0.942798
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,-0.986928,0.000187,...,1111.92,0.023402,0.299377,0.270825,0.152293,1,0.000180,0.013406,0.037532,0.942798


In [8]:
##### ANN torch calibration and estimation function with code data pipeline correction:
def ANN_function_torch_LBFGS_surface(data, x_cols, y_cols, ignore, batch = 2048, log_y = False, params=None):

    X, Y = data[x_cols].copy(), data[y_cols].copy()
    
    label_encoders_ivol = {}
    categorical_columns = []
    for col in categorical_columns:
        label_encoder = LabelEncoder()
        X[col] = label_encoder.fit_transform(X[col])
        label_encoders_ivol[col] = label_encoder

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size= 0.7, shuffle = True, random_state = 0)


    scaler_X_ivol = StandardScaler()

    non_categ_cols_ivol = [col for col in X_train.columns if col not in ignore]
    X_train_encoded_ivol_std = X_train.copy()
    X_test_encoded_ivol_std = X_test.copy()
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.fit_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])
    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.transform(X_test_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_train_ivol_std = np.log(Y_train)
        Y_test_ivol_std  = np.log(Y_test)
    else:
        Y_train_ivol_std = Y_train
        Y_test_ivol_std  = Y_test


    _p           = params or {}
    n_layers     = _p.get('n_layers',     3)
    n_units      = _p.get('n_units',      64)
    max_iter     = _p.get('max_iter',     20)
    history_size = _p.get('history_size', 1)


    class MLP(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            layers = [nn.Linear(n_features, n_units), nn.GELU()]
            for _ in range(n_layers - 1):
                layers += [nn.Linear(n_units, n_units), nn.GELU()]
            layers.append(nn.Linear(n_units, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x) if log_y else torch.nn.functional.softplus(self.net(x))

    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


    model = MLP(len(X_train_encoded_ivol_std.columns)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.LBFGS(
        model.parameters(),
        max_iter=max_iter,
        history_size=history_size,
        line_search_fn="strong_wolfe"
    )

    model.train()
    n = len(X_train_encoded_ivol_std)


    for i in range(0, n, batch):
        end = min(i + batch, n)
        xb = torch.as_tensor(X_train_encoded_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)
        yb = torch.as_tensor(Y_train_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)

        def closure():
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            return loss
        
        for ii in range(5):
            total_loss = 0
            loss_temp = optimizer.step(closure)

            total_loss += loss_temp



            print(f"Run {ii:03d} | Training Loss: {total_loss:.4f}")
    

    model.eval()
    with torch.no_grad():
        
        X_test_encoded_ivol_std_t = torch.from_numpy(X_test_encoded_ivol_std.values.astype(np.float32)).to(device)
        X_train_encoded_ivol_std_t = torch.from_numpy(X_train_encoded_ivol_std.values.astype(np.float32)).to(device)
        y_pred_ann_ivol = model(X_test_encoded_ivol_std_t).cpu().numpy()
        y_pred_ann_ivol_train = model(X_train_encoded_ivol_std_t).cpu().numpy()

    if log_y:
        y_pred_ann_ivol       = np.exp(y_pred_ann_ivol)
        y_pred_ann_ivol_train = np.exp(y_pred_ann_ivol_train)

    rmse_ann_ivol = np.sqrt(mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol))
    mse_ann_ivol = mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol)
    mape_ann_ivol = mean_absolute_percentage_error(Y_test.values.ravel(), y_pred_ann_ivol)
    rsquared_ann_ivol = r2_score(Y_test.values.ravel(), y_pred_ann_ivol)


    print("mse:", mse_ann_ivol)
    print("rmse:", rmse_ann_ivol)
    print("mape:", mape_ann_ivol)
    print("rsquared:", rsquared_ann_ivol)


    for col in categorical_columns:
        X_test_encoded_ivol_std[col] = label_encoders_ivol[col].inverse_transform(X_test_encoded_ivol_std[col])
        X_train_encoded_ivol_std[col] = label_encoders_ivol[col].inverse_transform(X_train_encoded_ivol_std[col])

    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_test_encoded_ivol_std[non_categ_cols_ivol])
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_test_ivol_std  = np.exp(Y_test_ivol_std)
        Y_train_ivol_std = np.exp(Y_train_ivol_std)

    output_ann_ivol = pd.concat([X_test_encoded_ivol_std, 
                                 Y_test_ivol_std.set_axis(X_test_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol, columns = ["output_ann"]).set_axis(X_test_encoded_ivol_std.index)], axis=1)
    output_ann_ivol_train = pd.concat([X_train_encoded_ivol_std, 
                                 Y_train_ivol_std.set_axis(X_train_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol_train, columns = ["output_ann"]).set_axis(X_train_encoded_ivol_std.index)], axis=1)


    output_ann_ivol["out_of_sample"] = 1
    output_ann_ivol_train["out_of_sample"] = 0

    output_ann_ivol_full = pd.concat([output_ann_ivol_train, output_ann_ivol], axis = 0)

    data_clean = data.drop(columns = [col for col in data.columns if col in output_ann_ivol_full.columns])
    data_out = pd.concat([data_clean, output_ann_ivol_full], axis = 1)

    data_out["mse_optim"] = mse_ann_ivol
    data_out["rmse_optim"] = rmse_ann_ivol
    data_out["mape_optim"] = mape_ann_ivol
    data_out["rsquared_optim"] = rsquared_ann_ivol

    return(data_out)

In [9]:
# Group data on day and cusip (underlying)
set_seed(42)
grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['impl_volatility']
ignore = ["cp_flag_one"]

streams = {}
results = {}
for i, group in grouped_data:
    streams[i] = torch.cuda.Stream()
    results[i] = torch.empty(1,device="cuda")

for i, group in grouped_data:
    stream = streams[i]
    with torch.cuda.stream(stream):
        print(group["date"].unique())
        
        temp = ANN_function_torch_LBFGS_surface(group, selected_columns_x, selected_columns_y, ignore, params=params_lbfgs_surface_ivol)
        results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


for stream in streams.values():
    stream.synchronize()

surface_ann_lbfgs_ivol_data = pd.concat(results.values())
surface_ann_lbfgs_ivol_data

_oos = surface_ann_lbfgs_ivol_data[surface_ann_lbfgs_ivol_data["out_of_sample"] == 1]
print("mse:", mean_squared_error(_oos["impl_volatility"], _oos["output_ann"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["impl_volatility"], _oos["output_ann"])))
print("mape:", mean_absolute_percentage_error(_oos["impl_volatility"], _oos["output_ann"]))
print("rsquared:", r2_score(_oos["impl_volatility"], _oos["output_ann"]))

surface_ann_lbfgs_ivol_data = surface_ann_lbfgs_ivol_data.rename(columns={"output_ann":"impl_vol_ann"})
surface_ann_lbfgs_ivol_data

<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]


C:\Users\chril\AppData\Local\Temp\ipykernel_23984\2280421579.py:110: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  yb = torch.as_tensor(Y_train_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)


Run 000 | Training Loss: 0.1472
Run 001 | Training Loss: 0.0003
Run 002 | Training Loss: 0.0001
Run 003 | Training Loss: 0.0000
Run 004 | Training Loss: 0.0000
mse: 8.735742332606544e-05
rmse: 0.009346519316091175
mape: 0.018246218909433957
rsquared: 0.9915295919673706
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.1327
Run 001 | Training Loss: 0.0003
Run 002 | Training Loss: 0.0001
Run 003 | Training Loss: 0.0000
Run 004 | Training Loss: 0.0000
mse: 3.6057238657726696e-05
rmse: 0.0060047679936635935
mape: 0.014133968610125974
rsquared: 0.9965476836013605
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.1208
Run 001 | Training Loss: 0.0004
Run 002 | Training Loss: 0.0001
Run 003 | Training Loss: 0.0001
Run 004 | Training Loss: 0.0000
mse: 4.222314064147399e-05
rmse: 0.006497933567025289
mape: 0.014576105437956954
rsquared: 0.9948674169344165
<DatetimeArray>
['2003-01-07 00:00:00']
Le

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,ddelta,dgamma,...,stock_price,realized_vol,moneyness,impl_volatility,impl_vol_ann,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.309251,0.047734,...,86.08,0.030519,0.044533,0.233642,0.233130,0,0.000087,0.009347,0.018246,0.991530
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.256302,0.045347,...,86.08,0.030519,0.055582,0.224751,0.228661,1,0.000087,0.009347,0.018246,0.991530
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.216934,0.041150,...,86.08,0.030519,0.066511,0.225981,0.224484,1,0.000087,0.009347,0.018246,0.991530
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.139982,0.032262,...,86.08,0.030519,0.088018,0.218488,0.217118,0,0.000087,0.009347,0.018246,0.991530
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.361403,0.037649,...,86.08,0.048298,0.044533,0.250788,0.250963,0,0.000087,0.009347,0.018246,0.991530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,-0.191240,0.001474,...,1111.92,0.093558,-0.131406,0.196156,0.196785,1,0.000016,0.004056,0.010601,0.994763
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,-0.220435,0.001657,...,1111.92,0.093558,-0.111101,0.189753,0.190702,0,0.000016,0.004056,0.010601,0.994763
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,-0.111764,0.000908,...,1111.92,0.093558,-0.211449,0.222397,0.221433,1,0.000016,0.004056,0.010601,0.994763
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,-0.986928,0.000187,...,1111.92,0.023402,0.299377,0.270825,0.298932,1,0.000016,0.004056,0.010601,0.994763


In [10]:
##### ANN torch calibration and estimation function with code data pipeline correction:
def ANN_function_torch_LBFGS_variable(data, x_cols, y_cols, ignore, model, criterion, optimizer, batch = 2048, log_y = False, params=None):

    X, Y = data[x_cols].copy(), data[y_cols].copy()
    
    label_encoders_ivol = {}
    categorical_columns = []
    for col in categorical_columns:
        label_encoder = LabelEncoder()
        X[col] = label_encoder.fit_transform(X[col])
        label_encoders_ivol[col] = label_encoder

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size= 0.7, shuffle = True, random_state = 0)


    scaler_X_ivol = StandardScaler()

    non_categ_cols_ivol = [col for col in X_train.columns if col not in ignore]
    X_train_encoded_ivol_std = X_train.copy()
    X_test_encoded_ivol_std = X_test.copy()
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.fit_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])
    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.transform(X_test_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_train_ivol_std = np.log(Y_train)
        Y_test_ivol_std  = np.log(Y_test)
    else:
        Y_train_ivol_std = Y_train
        Y_test_ivol_std  = Y_test




    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")




    model.train()
    n = len(X_train_encoded_ivol_std)


    for i in range(0, n, batch):
        end = min(i + batch, n)
        xb = torch.as_tensor(X_train_encoded_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)
        yb = torch.as_tensor(Y_train_ivol_std.iloc[i:end].values, device=device, dtype=torch.float32)

        def closure():
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            return loss

        for ii in range(5):
            total_loss = 0
            loss_temp = optimizer.step(closure)

            total_loss += loss_temp



            print(f"Run {ii:03d} | Training Loss: {total_loss:.4f}")
    

    model.eval()
    with torch.no_grad():
        
        X_test_encoded_ivol_std_t = torch.from_numpy(X_test_encoded_ivol_std.values.astype(np.float32)).to(device)
        X_train_encoded_ivol_std_t = torch.from_numpy(X_train_encoded_ivol_std.values.astype(np.float32)).to(device)
        y_pred_ann_ivol = model(X_test_encoded_ivol_std_t).cpu().numpy()
        y_pred_ann_ivol_train = model(X_train_encoded_ivol_std_t).cpu().numpy()

    if log_y:
        y_pred_ann_ivol       = np.exp(y_pred_ann_ivol)
        y_pred_ann_ivol_train = np.exp(y_pred_ann_ivol_train)

    rmse_ann_ivol = np.sqrt(mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol))
    mse_ann_ivol = mean_squared_error(Y_test.values.ravel(), y_pred_ann_ivol)
    mape_ann_ivol = mean_absolute_percentage_error(Y_test.values.ravel(), y_pred_ann_ivol)
    rsquared_ann_ivol = r2_score(Y_test.values.ravel(), y_pred_ann_ivol)


    print("mse:", mse_ann_ivol)
    print("rmse:", rmse_ann_ivol)
    print("mape:", mape_ann_ivol)
    print("rsquared:", rsquared_ann_ivol)


    for col in categorical_columns:
        X_test_encoded_ivol_std[col] = label_encoders_ivol[col].inverse_transform(X_test_encoded_ivol_std[col])
        X_train_encoded_ivol_std[col] = label_encoders_ivol[col].inverse_transform(X_train_encoded_ivol_std[col])

    X_test_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_test_encoded_ivol_std[non_categ_cols_ivol])
    X_train_encoded_ivol_std[non_categ_cols_ivol] = scaler_X_ivol.inverse_transform(X_train_encoded_ivol_std[non_categ_cols_ivol])

    if log_y:
        Y_test_ivol_std  = np.exp(Y_test_ivol_std)
        Y_train_ivol_std = np.exp(Y_train_ivol_std)

    output_ann_ivol = pd.concat([X_test_encoded_ivol_std, 
                                 Y_test_ivol_std.set_axis(X_test_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol, columns = ["output_ann"]).set_axis(X_test_encoded_ivol_std.index)], axis=1)
    output_ann_ivol_train = pd.concat([X_train_encoded_ivol_std, 
                                 Y_train_ivol_std.set_axis(X_train_encoded_ivol_std.index), 
                                 pd.DataFrame(y_pred_ann_ivol_train, columns = ["output_ann"]).set_axis(X_train_encoded_ivol_std.index)], axis=1)


    output_ann_ivol["out_of_sample"] = 1
    output_ann_ivol_train["out_of_sample"] = 0

    output_ann_ivol_full = pd.concat([output_ann_ivol_train, output_ann_ivol], axis = 0)

    data_clean = data.drop(columns = [col for col in data.columns if col in output_ann_ivol_full.columns])
    data_out = pd.concat([data_clean, output_ann_ivol_full], axis = 1)

    data_out["mse_optim"] = mse_ann_ivol
    data_out["rmse_optim"] = rmse_ann_ivol
    data_out["mape_optim"] = mape_ann_ivol
    data_out["rsquared_optim"] = rsquared_ann_ivol

    return(data_out)

In [16]:
# ANN variable LBGFS loop:
set_seed(42)

_p        = params_lbfgs_variable_ivol
_n_units  = _p.get('n_units',     64)
_n_layers = _p.get('n_layers',    3)
_max_iter = _p.get('max_iter',    20)
_hist_sz  = _p.get('history_size', 1)
device    = torch.device("cuda")


grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['impl_volatility']
ignore = ["cp_flag_one"]


_lrs = [nn.Linear(len(selected_columns_x), _n_units), nn.GELU()]
for _ in range(_n_layers - 1):
    _lrs += [nn.Linear(_n_units, _n_units), nn.GELU()]
_lrs.append(nn.Linear(_n_units, 1))
model = nn.Sequential(*_lrs).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(
    model.parameters(),
    max_iter=_max_iter,
    history_size=_hist_sz,
    line_search_fn="strong_wolfe"
)


results = {}

for i, group in grouped_data:
    print(group["date"].unique())
    
    temp = ANN_function_torch_LBFGS_variable(group, selected_columns_x, selected_columns_y, ignore, model=model, criterion=criterion, optimizer=optimizer)
    results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()



variable_ann_lbfgs_ivol_data = pd.concat(results.values())
variable_ann_lbfgs_ivol_data

_oos = variable_ann_lbfgs_ivol_data[variable_ann_lbfgs_ivol_data["out_of_sample"] == 1]
print("mse:", mean_squared_error(_oos["impl_volatility"], _oos["output_ann"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["impl_volatility"], _oos["output_ann"])))
print("mape:", mean_absolute_percentage_error(_oos["impl_volatility"], _oos["output_ann"]))
print("rsquared:", r2_score(_oos["impl_volatility"], _oos["output_ann"]))

variable_ann_lbfgs_ivol_data = variable_ann_lbfgs_ivol_data.rename(columns={"output_ann":"impl_vol_ann"})
variable_ann_lbfgs_ivol_data

<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.1849
Run 001 | Training Loss: 0.0027
Run 002 | Training Loss: 0.0009
Run 003 | Training Loss: 0.0003
Run 004 | Training Loss: 0.0002
mse: 0.0002307290066710268
rmse: 0.015189766511405854
mape: 0.033008882621907376
rsquared: 0.9776279020482082
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.0037
Run 001 | Training Loss: 0.0003
Run 002 | Training Loss: 0.0001
Run 003 | Training Loss: 0.0001
Run 004 | Training Loss: 0.0001
mse: 7.614917847103812e-05
rmse: 0.008726349664724542
mape: 0.02257897026111183
rsquared: 0.9927090629408982
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.0071
Run 001 | Training Loss: 0.0006
Run 002 | Training Loss: 0.0003
Run 003 | Training Loss: 0.0002
Run 004 | Training Loss: 0.0002
mse: 0.00023235991570601274
rmse: 0.015243356444891418
mape: 0.03323664934589461
r

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,ddelta,dgamma,...,stock_price,realized_vol,moneyness,impl_volatility,impl_vol_ann,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.309251,0.047734,...,86.08,0.030519,0.044533,0.233642,0.230486,0,0.000231,0.015190,0.033009,0.977628
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.256302,0.045347,...,86.08,0.030519,0.055582,0.224751,0.225330,1,0.000231,0.015190,0.033009,0.977628
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.216934,0.041150,...,86.08,0.030519,0.066511,0.225981,0.220404,1,0.000231,0.015190,0.033009,0.977628
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.139982,0.032262,...,86.08,0.030519,0.088018,0.218488,0.211213,0,0.000231,0.015190,0.033009,0.977628
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.361403,0.037649,...,86.08,0.048298,0.044533,0.250788,0.249270,0,0.000231,0.015190,0.033009,0.977628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,-0.191240,0.001474,...,1111.92,0.093558,-0.131406,0.196156,0.198107,1,0.000032,0.005684,0.016780,0.989715
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,-0.220435,0.001657,...,1111.92,0.093558,-0.111101,0.189753,0.192332,0,0.000032,0.005684,0.016780,0.989715
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,-0.111764,0.000908,...,1111.92,0.093558,-0.211449,0.222397,0.220180,1,0.000032,0.005684,0.016780,0.989715
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,-0.986928,0.000187,...,1111.92,0.023402,0.299377,0.270825,0.223693,1,0.000032,0.005684,0.016780,0.989715


Surface Estimation through Option Price modelling

In [3]:
# GPU Implied Vol estimator (Newton Rhapson method) with batching and parallelization

def bs_price_torch(S, K, T, r, cp, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    normal = torch.distributions.Normal(0.0, 1.0)
    return cp * (S * normal.cdf(cp * d1) - K * torch.exp(-r * T) * normal.cdf(cp * d2))

def bs_vega_torch(S, K, T, r, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    normal = torch.distributions.Normal(0.0, 1.0)
    return S * normal.log_prob(d1).exp() * torch.sqrt(T)


def implied_vol_newton_gpu_batched(
    dfs: list[pd.DataFrame],
    price_cols: list[str],
    batch_size: int = 10_000,
    initial_guess: float = 0.2,
    tol: float = 1e-10,
    max_iter: int = 100,
) -> list[torch.Tensor]:
    """
    Compute implied vol for N dataframes in a single batched GPU pass,
    processing `batch_size` rows at a time to respect GPU memory limits.

    Parameters
    ----------
    dfs        : list of DataFrames, all sharing the same index/length
    price_cols : price column name per DataFrame (same order as dfs)
    batch_size : number of rows per GPU batch (tune to VRAM budget)

    Returns
    -------
    List of 1-D tensors (one per DataFrame) with implied vols, on CPU.
    """
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np     = [df["stock_price"].values  for df in dfs]
    K_np     = [df["strike_price"].values for df in dfs]
    T_np     = [df["time_to_mat"].values  for df in dfs]
    r_np     = [df["rf"].values           for df in dfs]
    cp_np    = [df["cp_flag_one"].values  for df in dfs]
    price_np = [df[col].values for df, col in zip(dfs, price_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        prices = torch.stack(
            [torch.as_tensor(p[sl], dtype=dtype, device=device) for p in price_np],
            dim=0,
        )

        sigma = torch.full((N, batch_end - batch_start), initial_guess,
                           dtype=dtype, device=device)

        for _ in range(max_iter):
            price_est = bs_price_torch(S, K, T, r, cp, sigma)
            vega      = bs_vega_torch(S, K, T, r, sigma)

            diff      = price_est - prices
            converged = torch.abs(diff) < tol

            if torch.all(converged):
                break

            vega  = torch.clamp(vega, min=1e-12)
            sigma = sigma - diff / vega
            sigma = torch.clamp(sigma, min=1e-8, max=5.0)

        for i in range(N):
            results[i][sl] = sigma[i].cpu()

        del S, K, T, r, cp, prices, sigma, price_est, vega, diff, converged
        torch.cuda.empty_cache()

    return results


def bs_price_from_iv_gpu_batched(
    dfs: list[pd.DataFrame],
    iv_cols: list[str],
    batch_size: int = 10_000,
) -> list[torch.Tensor]:
    """
    Compute Black-Scholes prices from implied volatilities for N dataframes,
    processing `batch_size` rows at a time.

    Parameters
    ----------
    dfs     : list of DataFrames sharing the same index/length
    iv_cols : IV column name per DataFrame (same order as dfs)

    Returns
    -------
    List of 1-D tensors (one per DataFrame) with BS prices, on CPU.
    """
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np  = [df["stock_price"].values  for df in dfs]
    K_np  = [df["strike_price"].values for df in dfs]
    T_np  = [df["time_to_mat"].values  for df in dfs]
    r_np  = [df["rf"].values           for df in dfs]
    cp_np = [df["cp_flag_one"].values  for df in dfs]
    iv_np = [df[col].values for df, col in zip(dfs, iv_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        sigmas = torch.stack(
            [torch.as_tensor(iv[sl], dtype=dtype, device=device) for iv in iv_np],
            dim=0,
        )

        prices = bs_price_torch(S, K, T, r, cp, sigmas)

        for i in range(N):
            results[i][sl] = prices[i].cpu()

        del S, K, T, r, cp, sigmas, prices
        torch.cuda.empty_cache()

    return results

In [18]:
# Group data on day and cusip (underlying)
set_seed(42)
grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['price']

streams = {}
results = {}
for i, group in grouped_data:
    streams[i] = torch.cuda.Stream()
    results[i] = torch.empty(1,device="cuda")

for i, group in grouped_data:
    stream = streams[i]
    with torch.cuda.stream(stream):
        print(group["date"].unique())
        
        temp = XGB_function_surface(group, selected_columns_x, selected_columns_y, params=params_xgb_surface_price)
        results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


for stream in streams.values():
    stream.synchronize()

surface_xgb_price_data = pd.concat(results.values())
surface_xgb_price_data

_oos = surface_xgb_price_data[surface_xgb_price_data["out_of_sample"] == 1]
print("mse:", mean_squared_error(_oos["price"], _oos["output_xgb"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["price"], _oos["output_xgb"])))
print("mape:", mean_absolute_percentage_error(_oos["price"], _oos["output_xgb"]))
print("rsquared:", r2_score(_oos["price"], _oos["output_xgb"]))

surface_xgb_price_data = surface_xgb_price_data.rename(columns={"output_xgb":"price_xgb"})
surface_xgb_price_data

<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 1082.26806640625
rmse: 32.89784288378571
mape: 1.8063726425170898
rsquared: 0.9040464162826538
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 81.2040023803711
rmse: 9.011326338579194
mape: 0.6717473864555359
rsquared: 0.9891369938850403
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 130.86659240722656
rmse: 11.439693719992095
mape: 0.8617540001869202
rsquared: 0.9866800308227539
<DatetimeArray>
['2003-01-07 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 279.05841064453125
rmse: 16.70504147389438
mape: 0.9343326091766357
rsquared: 0.973307192325592
<DatetimeArray>
['2003-01-08 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 106.22261047363281
rmse: 10.306435391231675
mape: 1.118148922920227
rsquared: 0.9856283664703369
<DatetimeArray>
['2003-01-09 00:00:00']
Length: 1, dtype: datetime64[ms]
mse: 142.76449584960938
rmse: 11.94840976237463
mape: 1.

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,stock_price,realized_vol,moneyness,price,price_xgb,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.233642,0.309251,...,86.08,0.030519,0.044533,1.400,-2.720566,0,1082.268066,32.897843,1.806373,0.904046
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.224751,0.256302,...,86.08,0.030519,0.055582,1.050,-3.884384,1,1082.268066,32.897843,1.806373,0.904046
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.225981,0.216934,...,86.08,0.030519,0.066511,0.850,-3.884384,1,1082.268066,32.897843,1.806373,0.904046
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.218488,0.139982,...,86.08,0.030519,0.088018,0.475,-5.893448,0,1082.268066,32.897843,1.806373,0.904046
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.250788,0.361403,...,86.08,0.048298,0.044533,2.300,0.022400,0,1082.268066,32.897843,1.806373,0.904046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,0.196156,-0.191240,...,1111.92,0.093558,-0.131406,21.100,20.967663,1,190.018661,13.784726,1.247661,0.961253
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,0.189753,-0.220435,...,1111.92,0.093558,-0.111101,24.500,23.015650,0,190.018661,13.784726,1.247661,0.961253
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,0.222397,-0.111764,...,1111.92,0.093558,-0.211449,12.300,9.794445,1,190.018661,13.784726,1.247661,0.961253
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,0.270825,-0.986928,...,1111.92,0.023402,0.299377,388.500,273.891052,1,190.018661,13.784726,1.247661,0.961253


In [19]:
### LBFGS Torch ANN surface estimation through Option price modelling
set_seed(42)
grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['price']
ignore = ["cp_flag_one"]

streams = {}
results = {}
for i, group in grouped_data:
    streams[i] = torch.cuda.Stream()
    results[i] = torch.empty(1,device="cuda")

for i, group in grouped_data:
    stream = streams[i]
    with torch.cuda.stream(stream):
        print(group["date"].unique())
        
        temp = ANN_function_torch_LBFGS_surface(group, selected_columns_x, selected_columns_y, ignore, log_y=True, params=params_lbfgs_surface_price)
        results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


for stream in streams.values():
    stream.synchronize()

surface_ann_lbfgs_price_data = pd.concat(results.values())
surface_ann_lbfgs_price_data

_oos = surface_ann_lbfgs_price_data[surface_ann_lbfgs_price_data["out_of_sample"] == 1]
print("mse:", mean_squared_error(_oos["price"], _oos["output_ann"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["price"], _oos["output_ann"])))
print("mape:", mean_absolute_percentage_error(_oos["price"], _oos["output_ann"]))
print("rsquared:", r2_score(_oos["price"], _oos["output_ann"]))

surface_ann_lbfgs_price_data = surface_ann_lbfgs_price_data.rename(columns={"output_ann":"price_ann"})
surface_ann_lbfgs_price_data

<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 13.4325
Run 001 | Training Loss: 0.0365
Run 002 | Training Loss: 0.0056
Run 003 | Training Loss: 0.0027
Run 004 | Training Loss: 0.0017
mse: 10.50755388043554
rmse: 3.241535728699522
mape: 0.03391736883154761
rsquared: 0.9990684034020044
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 12.0657
Run 001 | Training Loss: 0.0257
Run 002 | Training Loss: 0.0082
Run 003 | Training Loss: 0.0036
Run 004 | Training Loss: 0.0025
mse: 27.199087898507045
rmse: 5.21527447969012
mape: 0.036046501950864035
rsquared: 0.99636146482467
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 12.8670
Run 001 | Training Loss: 0.0296
Run 002 | Training Loss: 0.0078
Run 003 | Training Loss: 0.0038
Run 004 | Training Loss: 0.0023
mse: 15.413563087775083
rmse: 3.9260110911426476
mape: 0.0424648986437505
rsquared: 0.998431165

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,stock_price,realized_vol,moneyness,price,price_ann,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.233642,0.309251,...,86.08,0.030519,0.044533,1.400,1.379450,0,10.507554,3.241536,0.033917,0.999068
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.224751,0.256302,...,86.08,0.030519,0.055582,1.050,1.080765,1,10.507554,3.241536,0.033917,0.999068
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.225981,0.216934,...,86.08,0.030519,0.066511,0.850,0.834318,1,10.507554,3.241536,0.033917,0.999068
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.218488,0.139982,...,86.08,0.030519,0.088018,0.475,0.477074,0,10.507554,3.241536,0.033917,0.999068
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.250788,0.361403,...,86.08,0.048298,0.044533,2.300,2.302574,0,10.507554,3.241536,0.033917,0.999068
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,0.196156,-0.191240,...,1111.92,0.093558,-0.131406,21.100,21.511543,1,25.067427,5.006738,0.037546,0.994889
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,0.189753,-0.220435,...,1111.92,0.093558,-0.111101,24.500,24.965252,0,25.067427,5.006738,0.037546,0.994889
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,0.222397,-0.111764,...,1111.92,0.093558,-0.211449,12.300,12.271599,1,25.067427,5.006738,0.037546,0.994889
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,0.270825,-0.986928,...,1111.92,0.023402,0.299377,388.500,338.564056,1,25.067427,5.006738,0.037546,0.994889


In [22]:
# ANN torch LBGFS price
set_seed(42)

_p        = params_lbfgs_variable_price
_n_units  = _p.get('n_units',     64)
_n_layers = _p.get('n_layers',    3)
_max_iter = _p.get('max_iter',    20)
_hist_sz  = _p.get('history_size', 1)
device    = torch.device("cuda")


grouped_data = data.groupby(['date'])
selected_columns_x = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
selected_columns_y = ['price']
ignore = ["cp_flag_one"]


_lrs = [nn.Linear(len(selected_columns_x), _n_units), nn.GELU()]
for _ in range(_n_layers - 1):
    _lrs += [nn.Linear(_n_units, _n_units), nn.GELU()]
_lrs.append(nn.Linear(_n_units, 1))
model = nn.Sequential(*_lrs).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(
    model.parameters(),
    max_iter=_max_iter,
    history_size=_hist_sz,
    line_search_fn="strong_wolfe"
)


results = {}

for i, group in grouped_data:
    print(group["date"].unique())
    
    temp = ANN_function_torch_LBFGS_variable(group, selected_columns_x, selected_columns_y, ignore, model=model, criterion=criterion, optimizer=optimizer, log_y=True)
    results[i] = temp.copy()
    del temp, group
    torch.cuda.synchronize()
    torch.cuda.empty_cache()



variable_ann_lbfgs_price_data = pd.concat(results.values())
variable_ann_lbfgs_price_data

_oos = variable_ann_lbfgs_price_data[variable_ann_lbfgs_price_data["out_of_sample"] == 1]
print("mse:", mean_squared_error(_oos["price"], _oos["output_ann"]))
print("rmse:", np.sqrt(mean_squared_error(_oos["price"], _oos["output_ann"])))
print("mape:", mean_absolute_percentage_error(_oos["price"], _oos["output_ann"]))
print("rsquared:", r2_score(_oos["price"], _oos["output_ann"]))

variable_ann_lbfgs_price_data = variable_ann_lbfgs_price_data.rename(columns={"output_ann":"price_ann"})
variable_ann_lbfgs_price_data

<DatetimeArray>
['2003-01-02 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 12.4499
Run 001 | Training Loss: 1.4876
Run 002 | Training Loss: 0.2008
Run 003 | Training Loss: 0.0579
Run 004 | Training Loss: 0.0271
mse: 190.0277071655713
rmse: 13.78505375998118
mape: 0.10865462022112665
rsquared: 0.983152200071039
<DatetimeArray>
['2003-01-03 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.2603
Run 001 | Training Loss: 0.0147
Run 002 | Training Loss: 0.0112
Run 003 | Training Loss: 0.0089
Run 004 | Training Loss: 0.0062
mse: 32.858221162218825
rmse: 5.732209099659469
mape: 0.04577333061113538
rsquared: 0.9956044190178867
<DatetimeArray>
['2003-01-06 00:00:00']
Length: 1, dtype: datetime64[ms]
Run 000 | Training Loss: 0.2821
Run 001 | Training Loss: 0.0086
Run 002 | Training Loss: 0.0057
Run 003 | Training Loss: 0.0042
Run 004 | Training Loss: 0.0036
mse: 17.630964827723194
rmse: 4.198924246485425
mape: 0.057914694625427886
rsquared: 0.998205471

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,stock_price,realized_vol,moneyness,price,price_ann,out_of_sample,mse_optim,rmse_optim,mape_optim,rsquared_optim
0,2003-01-02,2003-02-22,C,90.0,1.30,1.50,1464.0,5083.0,0.233642,0.309251,...,86.08,0.030519,0.044533,1.400,1.429631,0,190.027707,13.785054,0.108655,0.983152
1,2003-01-02,2003-02-22,C,91.0,0.95,1.15,255.0,540.0,0.224751,0.256302,...,86.08,0.030519,0.055582,1.050,1.139008,1,190.027707,13.785054,0.108655,0.983152
2,2003-01-02,2003-02-22,C,92.0,0.75,0.95,0.0,355.0,0.225981,0.216934,...,86.08,0.030519,0.066511,0.850,0.901695,1,190.027707,13.785054,0.108655,0.983152
3,2003-01-02,2003-02-22,C,94.0,0.40,0.55,0.0,835.0,0.218488,0.139982,...,86.08,0.030519,0.088018,0.475,0.557500,0,190.027707,13.785054,0.108655,0.983152
4,2003-01-02,2003-03-22,C,90.0,2.15,2.45,2028.0,13151.0,0.250788,0.361403,...,86.08,0.048298,0.044533,2.300,2.194749,0,190.027707,13.785054,0.108655,0.983152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163837,2003-12-31,2004-09-18,P,975.0,20.10,22.10,0.0,744.0,0.196156,-0.191240,...,1111.92,0.093558,-0.131406,21.100,21.417723,1,1.959831,1.399940,0.022443,0.999600
163838,2003-12-31,2004-09-18,P,995.0,23.50,25.50,607.0,4679.0,0.189753,-0.220435,...,1111.92,0.093558,-0.111101,24.500,24.954203,0,1.959831,1.399940,0.022443,0.999600
163839,2003-12-31,2004-09-18,P,900.0,11.50,13.10,3150.0,4822.0,0.222397,-0.111764,...,1111.92,0.093558,-0.211449,12.300,12.256930,1,1.959831,1.399940,0.022443,0.999600
163840,2003-12-31,2004-03-20,P,1500.0,387.50,389.50,0.0,13601.0,0.270825,-0.986928,...,1111.92,0.023402,0.299377,388.500,377.236298,1,1.959831,1.399940,0.022443,0.999600


In [ ]:
# Implied vol from price for whole data models
price_dfs  = [output_xgb_price, output_ann_price_lbfgs]
price_cols = ["price_xgb", "price_ann"]

iv_xgb, iv_ann_lbfgs = implied_vol_newton_gpu_batched(
    dfs=price_dfs,
    price_cols=price_cols,
    batch_size=10_000,
)

output_xgb_price["BS_impl_vol_xgb"]         = iv_xgb.numpy()
output_ann_price_lbfgs["BS_impl_vol_ann"]   = iv_ann_lbfgs.numpy()

# Price from ivol for whole data models
iv_dfs  = [output_xgb_ivol, output_ann_ivol_lbfgs]
iv_cols = ["impl_vol_xgb", "impl_vol_ann"]

p_xgb, p_ann_lbfgs = bs_price_from_iv_gpu_batched(
    dfs=iv_dfs,
    iv_cols=iv_cols,
    batch_size=10_000,
)

output_xgb_ivol["BS_price_xgb"] = p_xgb.numpy()
output_ann_ivol_lbfgs["BS_price_ann"]  = p_ann_lbfgs.numpy()

In [ ]:
# Implied vol from price for partial data models
price_dfs  = [surface_xgb_price_data, surface_ann_lbfgs_price_data, variable_ann_lbfgs_price_data]
price_cols = ["price_xgb", "price_ann", "price_ann"]

iv_xgb_s, iv_ann_lbgfs_s, iv_ann_lbgfs_v = implied_vol_newton_gpu_batched(
    dfs=price_dfs,
    price_cols=price_cols,
    batch_size=10_000,
)

surface_xgb_price_data["BS_impl_vol_xgb"] = iv_xgb_s.numpy()
surface_ann_lbfgs_price_data["BS_impl_vol_ann"]        = iv_ann_lbgfs_s.numpy()
variable_ann_lbfgs_price_data["BS_impl_vol_ann"]        = iv_ann_lbgfs_v.numpy()

# Price from ivol for partial data models
iv_dfs  = [surface_xgb_ivol_data, surface_ann_lbfgs_ivol_data, variable_ann_lbfgs_ivol_data]
iv_cols = ["impl_vol_xgb", "impl_vol_ann", "impl_vol_ann"]

p_xgb_s, p_ann_lbgfs_s, p_ann_lbgfs_v = bs_price_from_iv_gpu_batched(
    dfs=iv_dfs,
    iv_cols=iv_cols,
    batch_size=10_000,
)

surface_xgb_price_data["BS_price_xgb"] = p_xgb_s.numpy()
surface_ann_lbfgs_price_data["BS_price_ann"]  = p_ann_lbgfs_s.numpy()
variable_ann_lbfgs_price_data["BS_price_ann"]  = p_ann_lbgfs_v.numpy()

In [14]:
# Whole data models data output for trading strategies application

outputpath1 = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\ivol_xgb_output_2002.dta"
outputpath3 = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\ivol_ann_lbfgs_output_2002.dta"
outputpath4 = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\price_xgb_output_2002.dta"
outputpath6 = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\price_ann_lbfgs_output_2002.dta"


output_xgb_ivol.to_stata(outputpath1)
output_ann_ivol_lbfgs.to_stata(outputpath3)
output_xgb_price.to_stata(outputpath4)
output_ann_price_lbfgs.to_stata(outputpath6)


In [24]:
# Surface and variable data output for trading strategies application


outputpath1 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\surface_xgb_ivol_data_2003.dta"
outputpath4 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\surface_ann_lbfgs_ivol_data_2003.dta"
outputpath5 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\variable_ann_lbfgs_ivol_data_2003.dta"
outputpath6 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\surface_xgb_price_data_2003.dta"
outputpath9 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\surface_ann_lbfgs_price_data_2003.dta"
outputpath10 = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\variable_ann_lbfgs_price_data_2003.dta"

surface_xgb_ivol_data.to_stata(outputpath1)
surface_ann_lbfgs_ivol_data.to_stata(outputpath4)
variable_ann_lbfgs_ivol_data.to_stata(outputpath5)
surface_xgb_price_data.to_stata(outputpath6)
surface_ann_lbfgs_price_data.to_stata(outputpath9)
variable_ann_lbfgs_price_data.to_stata(outputpath10)
